# Autoencoder GRU, NSL-KDD

Autoencoder secuencial que reconstruye ventanas de W = 5 registros consecutivos en el orden original del fichero. Se entrena con KDDTrain+_20Percent y se evalúa con KDDTest+, que incluye tipos de ataque no vistos en entrenamiento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
BASE = Path.home() / ".cache/kagglehub/datasets/hassan06/nslkdd/versions/1"
df_train = pd.read_csv(BASE / "KDDTrain+_20Percent.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])
df_test = pd.read_csv(BASE / "KDDTest+.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])

df_train['is_attack'] = (df_train['label'] != 'normal').astype(int)
df_test['is_attack'] = (df_test['label'] != 'normal').astype(int)
df_train['_split'] = 'train'
df_test['_split'] = 'test'
df_all = pd.concat([df_train, df_test], ignore_index=True)

# One-hot conjunto de train y test, porque hay valores de service exclusivos de cada fichero
X_all = pd.get_dummies(df_all.drop(columns=['label', 'is_attack', '_split']), columns=['protocol_type', 'service', 'flag'])
is_train = (df_all['_split'] == 'train').to_numpy()

X_train_df = X_all[is_train].reset_index(drop=True)
X_test_df = X_all[~is_train].reset_index(drop=True)
y_train = df_train['is_attack'].to_numpy()
y_test = df_test['is_attack'].to_numpy()
tipos_nuevos = set(df_test.loc[df_test['is_attack'] == 1, 'label'].unique()) - set(df_train.loc[df_train['is_attack'] == 1, 'label'].unique())
print(f"X_train: {X_train_df.shape}, X_test: {X_test_df.shape}")
print(f"Ataques train: {y_train.sum()}/{len(y_train)}, ataques test: {y_test.sum()}/{len(y_test)}")
print(f"Tipos de ataque en test nunca vistos en train: {len(tipos_nuevos)} -> {sorted(tipos_nuevos)}")

X_train: (25192, 119), X_test: (22544, 119)
Ataques train: 11743/25192, ataques test: 12833/22544
Tipos de ataque en test nunca vistos en train: 18 -> ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'perl', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


In [3]:
def build_windows_traintest(X_train_df, y_train, X_test_df, y_test, W, stride=1):
    scaler = MinMaxScaler().fit(X_train_df.to_numpy())
    Xtr = scaler.transform(X_train_df.to_numpy()).astype('float32')
    Xte = scaler.transform(X_test_df.to_numpy()).astype('float32')

    train_windows = []
    n_tr = len(Xtr) - W + 1
    if n_tr > 0:
        cums = np.concatenate([[0], np.cumsum(y_train)])
        window_sums = cums[W:] - cums[:-W]
        for i in np.where(window_sums == 0)[0][::stride]:
            train_windows.append(Xtr[i:i + W])

    test_windows, test_labels = [], []
    n_te = len(Xte) - W + 1
    if n_te > 0:
        cums = np.concatenate([[0], np.cumsum(y_test)])
        window_sums = cums[W:] - cums[:-W]
        for i in range(0, n_te, stride):
            test_windows.append(Xte[i:i + W])
            test_labels.append(int(window_sums[i] > 0))

    X_train_w = np.array(train_windows) if train_windows else np.empty((0, W, Xtr.shape[1]))
    X_test_w = np.array(test_windows) if test_windows else np.empty((0, W, Xte.shape[1]))
    y_test_w = np.array(test_labels)
    return X_train_w, X_test_w, y_test_w


W = 5
STRIDE = 1
X_train_w, X_test_w, y_test_w = build_windows_traintest(X_train_df, y_train, X_test_df, y_test, W, stride=STRIDE)
print(f"Ventanas train: {X_train_w.shape}, ventanas test: {X_test_w.shape}, positivos test: {y_test_w.sum()}/{len(y_test_w)}")

Ventanas train: (1059, 5, 119), ventanas test: (22540, 5, 119), positivos test: 22267/22540


In [4]:
input_dim = X_train_w.shape[2]
bottleneck = 16

inputs = keras.Input(shape=(W, input_dim))
x = layers.GRU(32, return_sequences=True)(inputs)
x = layers.GRU(bottleneck, return_sequences=False)(x)
x = layers.RepeatVector(W)(x)
x = layers.GRU(bottleneck, return_sequences=True)(x)
x = layers.GRU(32, return_sequences=True)(x)
outputs = layers.TimeDistributed(layers.Dense(input_dim, activation='sigmoid'))(x)

autoencoder = keras.Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')

history = autoencoder.fit(
    X_train_w, X_train_w,
    epochs=100, batch_size=64, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=0,
)
print(f"Epocas entrenadas: {len(history.history['loss'])}, val_loss final: {history.history['val_loss'][-1]:.5f}")

Epocas entrenadas: 100, val_loss final: 0.01768


In [5]:
def reconstruction_error(X):
    return np.mean(np.square(X - autoencoder.predict(X, verbose=0)), axis=(1, 2))

threshold = np.percentile(reconstruction_error(X_train_w), 95)
err_test = reconstruction_error(X_test_w)
y_pred = (err_test > threshold).astype(int)

print(classification_report(y_test_w, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test_w, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test_w, err_test):.4f}")

              precision    recall  f1-score   support

      normal       0.05      0.98      0.09       273
      ataque       1.00      0.75      0.86     22267

    accuracy                           0.76     22540
   macro avg       0.52      0.87      0.47     22540
weighted avg       0.99      0.76      0.85     22540

[[  267     6]
 [ 5458 16809]]
ROC-AUC: 0.9545
